In [0]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load the data
df = spark.table("dataanalytics.ml1.loan_book_dirty").toPandas()

# ============================================================================
# A. DATA TYPE FIXES
# ============================================================================

# Standardize application_date - handle three formats
def parse_mixed_dates(date_str):
    if pd.isna(date_str):
        return pd.NaT
    try:
        # Try format 1: MM/DD/YYYY
        return pd.to_datetime(date_str, format='%m/%d/%Y')
    except:
        try:
            # Try format 2: YYYY-MM-DD
            return pd.to_datetime(date_str, format='%Y-%m-%d')
        except:
            try:
                # Try format 3: DD-MMM-YYYY
                return pd.to_datetime(date_str, format='%d-%b-%Y')
            except:
                return pd.NaT

df['application_date'] = df['application_date'].apply(parse_mixed_dates)

# Convert num_open_accounts from double to integer (after handling nulls later)
# Round age to nearest whole number
df['age'] = df['age'].round(0)

# ============================================================================
# B. DEDUPLICATION
# ============================================================================

# Keep first occurrence of each applicant_id_hash
df = df.drop_duplicates(subset='applicant_id_hash', keep='first')

# ============================================================================
# C. CATEGORICAL STANDARDIZATION
# ============================================================================

# Standardize loan_purpose - consolidate into 7 categories
loan_purpose_map = {
    'debt_consolidation': 'DEBT_CONSOLIDATION',
    'DEBT_CONSOLIDATION': 'DEBT_CONSOLIDATION',
    'debt consolidation': 'DEBT_CONSOLIDATION',
    'credit_card': 'CREDIT_CARD',
    'credit card': 'CREDIT_CARD',
    'CREDIT_CARD': 'CREDIT_CARD',
    'home_improvement': 'HOME_IMPROVEMENT',
    'HOME_IMPROVEMENT': 'HOME_IMPROVEMENT',
    'home improvement': 'HOME_IMPROVEMENT',
    'major_purchase': 'MAJOR_PURCHASE',
    'major purchase': 'MAJOR_PURCHASE',
    'MAJOR_PURCHASE': 'MAJOR_PURCHASE',
    'medical': 'MEDICAL',
    'MEDICAL': 'MEDICAL',
    'car': 'CAR',
    'CAR': 'CAR',
    'auto': 'CAR',
    'AUTO': 'CAR',
    'vehicle': 'CAR',
    'small_business': 'BUSINESS',
    'SMALL_BUSINESS': 'BUSINESS',
    'business': 'BUSINESS',
    'BUSINESS': 'BUSINESS'
}

# Map known values, set rest to OTHER
df['loan_purpose'] = df['loan_purpose'].map(loan_purpose_map).fillna('OTHER')

# Standardize home_ownership - consolidate into 4 categories
home_ownership_map = {
    'MORTGAGE': 'MORTGAGE',
    'mortgage': 'MORTGAGE',
    'Mortgage': 'MORTGAGE',
    'RENT': 'RENT',
    'rent': 'RENT',
    'Rent': 'RENT',
    'renting': 'RENT',
    'OWN': 'OWN',
    'own': 'OWN',
    'Own': 'OWN',
    'owned': 'OWN',
    'OWNED': 'OWN'
}

# Map known values, set rest to OTHER
df['home_ownership'] = df['home_ownership'].map(home_ownership_map).fillna('OTHER')

# ============================================================================
# D. MISSING VALUE TREATMENT
# ============================================================================

# annual_income
df['income_missing'] = df['annual_income'].isna()
annual_income_median = df['annual_income'].median()
df['annual_income'] = df['annual_income'].fillna(annual_income_median)

# months_since_last_delinquency
df['has_delinquency_history'] = df['months_since_last_delinquency'].notna()
df['months_since_last_delinquency'] = df['months_since_last_delinquency'].fillna(-1)
# Recode -1 to 999
df.loc[df['months_since_last_delinquency'] == -1, 'months_since_last_delinquency'] = 999

# employment_length_years
df['employment_missing'] = df['employment_length_years'].isna()
df['employment_length_years'] = df['employment_length_years'].fillna(0)

# num_open_accounts
df['num_accounts_missing'] = df['num_open_accounts'].isna()
num_accounts_median = df['num_open_accounts'].median()
df['num_open_accounts'] = df['num_open_accounts'].fillna(num_accounts_median)
# Now convert to integer
df['num_open_accounts'] = df['num_open_accounts'].astype(int)

# ============================================================================
# E. DATA QUALITY FIXES
# ============================================================================

# Cap credit_utilisation_pct at 100%
df['credit_utilisation_pct'] = df['credit_utilisation_pct'].clip(upper=100)

# Ensure all month columns are non-negative
df['months_since_oldest_account'] = df['months_since_oldest_account'].clip(lower=0)
df['months_at_current_address'] = df['months_at_current_address'].clip(lower=0)
# months_since_last_delinquency already handled (999 for no history)

# ============================================================================
# F. OUTLIER TREATMENT
# ============================================================================

# Cap annual_income at 99th percentile
income_99th = df['annual_income'].quantile(0.99)
df['annual_income'] = df['annual_income'].clip(upper=income_99th)

# Cap loan_amount at 99th percentile
loan_99th = df['loan_amount'].quantile(0.99)
df['loan_amount'] = df['loan_amount'].clip(upper=loan_99th)

# Cap total_revolving_balance at 99th percentile
balance_99th = df['total_revolving_balance'].quantile(0.99)
df['total_revolving_balance'] = df['total_revolving_balance'].clip(upper=balance_99th)

# ============================================================================
# G. FEATURE ENGINEERING FOR INTERPRETABILITY
# ============================================================================

# Create delinquency_category
def categorize_delinquency(months):
    if months == 999:
        return 'No_History'
    elif months <= 12:
        return 'Recent_0-12mo'
    elif months <= 36:
        return 'Moderate_12-36mo'
    else:
        return 'Old_36mo+'

df['delinquency_category'] = df['months_since_last_delinquency'].apply(categorize_delinquency)

# Create incomplete_observation_window flag
df['incomplete_observation_window'] = df['application_date'] > pd.Timestamp('2022-12-31')

# ============================================================================
# H. TRAIN/TEST SPLIT INTEGRITY
# ============================================================================

# Verify no applicant_id_hash appears in both train and test
train_ids = set(df[df['set'] == 'train']['applicant_id_hash'])
test_ids = set(df[df['set'] == 'test']['applicant_id_hash'])
overlap = train_ids.intersection(test_ids)

if len(overlap) > 0:
    # If overlap exists, keep records in their current set
    pass  # Deduplication already handled this by keeping first occurrence

# ============================================================================
# I. FINAL VALIDATION
# ============================================================================

# Convert age to integer
df['age'] = df['age'].astype(int)

# Validation checks
duplicates = df['applicant_id_hash'].duplicated().sum()
critical_nulls = df[['annual_income', 'employment_length_years', 'num_open_accounts', 'months_since_last_delinquency']].isna().sum().sum()
max_credit_util = df['credit_utilisation_pct'].max()
min_months_oldest = df['months_since_oldest_account'].min()
min_months_address = df['months_at_current_address'].min()
min_delinquency = df['months_since_last_delinquency'].min()

# Save to Spark table
spark_df = spark.createDataFrame(df)
spark_df.write.mode('overwrite').saveAsTable('dataanalytics.ml1.loan_book_silver_v2')

# Display final result
display(spark_df)

applicant_id_hash,age,annual_income,employment_length_years,home_ownership,region,num_open_accounts,num_delinquencies_2yr,total_revolving_balance,credit_utilisation_pct,months_since_oldest_account,num_hard_inquiries_6mo,loan_amount,interest_rate,loan_purpose,dti_ratio,months_since_last_delinquency,pct_accounts_current,application_date,application_dow,branch_code_id,months_at_current_address,email_domain_type,phone_verified,default_flag,set,income_missing,has_delinquency_history,employment_missing,num_accounts_missing,delinquency_category,incomplete_observation_window
11a2f242b28a331c,36,29401.0,3.6,MORTGAGE,North-Urban,9,7,1778.0,12.7,177.0,2,16326.0,21.11,MAJOR_PURCHASE,0.245,1.0,50.9,2021-06-14T00:00:00.000Z,Monday,347,119,other,true,0,test,false,true,false,false,Recent_0-12mo,false
5afff059dc04c6f0,25,32005.0,1.4,MORTGAGE,South-Urban,10,0,0.0,31.2,42.0,1,8293.0,12.69,MEDICAL,0.337,105.0,93.0,2021-08-28T00:00:00.000Z,Friday,367,7,free,false,0,train,false,true,false,false,Old_36mo+,false
eb85b183be0505f3,40,26730.0,0.0,MORTGAGE,East-Urban,9,0,3690.0,29.4,148.0,0,13080.0,10.45,DEBT_CONSOLIDATION,0.124,999.0,87.9,2021-02-23T00:00:00.000Z,Saturday,895,142,free,true,0,test,false,false,true,false,No_History,false
7d99629563bdb528,31,60105.0,7.5,MORTGAGE,South-Suburban,7,0,7636.0,48.8,115.0,2,6752.0,15.26,OTHER,0.218,58.0,96.1,2021-10-07T00:00:00.000Z,Thursday,387,69,other,true,1,test,false,true,false,false,Old_36mo+,false
658f1f05e484ce14,49,112275.0,6.9,MORTGAGE,East-Suburban,10,0,14450.0,68.2,324.0,3,19144.0,9.04,OTHER,0.195,999.0,90.8,2021-08-23T00:00:00.000Z,Sunday,561,182,free,true,0,test,false,false,false,false,No_History,false
57d7f15380846079,24,25010.0,4.0,MORTGAGE,North-Urban,6,1,3962.0,70.8,55.0,0,22065.0,16.56,DEBT_CONSOLIDATION,0.294,96.0,84.5,null,Thursday,272,164,corporate,true,1,train,false,true,false,false,Old_36mo+,false
dff5de728ed18812,52,39710.0,7.9,MORTGAGE,East-Urban,6,1,1630.0,66.6,281.0,2,9115.0,13.22,HOME_IMPROVEMENT,0.214,37.0,57.2,2021-12-30T00:00:00.000Z,Wednesday,354,13,free,true,0,train,false,true,false,false,Old_36mo+,false
64a84d3b67c218ea,39,94479.0,2.3,MORTGAGE,West-Urban,8,0,6899.0,30.4,156.0,4,14302.0,10.11,HOME_IMPROVEMENT,0.145,999.0,93.6,2022-01-12T00:00:00.000Z,Sunday,334,138,corporate,true,0,test,false,false,false,false,No_History,false
11011b26593b4559,48,49147.0,12.3,RENT,Central-Urban,6,0,4414.0,57.3,237.0,5,6509.0,11.55,MEDICAL,0.137,999.0,92.5,2021-04-08T00:00:00.000Z,Wednesday,431,75,corporate,true,0,train,false,false,false,false,No_History,false
2fffdaa2a3fadc3b,45,100583.0,3.4,RENT,North-Suburban,8,0,2985.0,33.7,224.0,3,11536.0,8.12,DEBT_CONSOLIDATION,0.05,999.0,78.4,2023-10-22T00:00:00.000Z,Monday,592,151,free,true,0,train,false,false,false,false,No_History,true
